# UrbanShift Churn Model Pipeline

End-to-end pipeline from the four raw CSV files to the final churn model table. Cleaning and feature engineering are performed in-memory. The only saved output is `customer_churn_model_table.csv`.

## 1. Setup

Update `DATA_DIR` if your raw CSV files are stored somewhere else.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DATA_DIR = Path("../data")          # Folder containing raw customers.csv, couriers.csv, deliveries.csv, incidents.csv
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "customers": DATA_DIR / "customers.csv",
    "couriers": DATA_DIR / "couriers.csv",
    "deliveries": DATA_DIR / "deliveries.csv",
    "incidents": DATA_DIR / "incidents.csv",
}

CHURN_CONFIG = {
    "lookback_days": 60,
    "prediction_horizon_days": 60,
    "snapshot_frequency": "month_end",
    "min_baseline_deliveries": 10,
    "relative_volume_drop": 0.30,
    "relative_revenue_drop": 0.30,
    "critical_volume_drop": 0.50,   # audit/reporting only; not exported as a feature
    "critical_revenue_drop": 0.50,  # audit/reporting only; not exported as a feature
}

CHURN_CONFIG

{'lookback_days': 60,
 'prediction_horizon_days': 60,
 'snapshot_frequency': 'month_end',
 'min_baseline_deliveries': 10,
 'relative_volume_drop': 0.3,
 'relative_revenue_drop': 0.3,
 'critical_volume_drop': 0.5,
 'critical_revenue_drop': 0.5}

## 2. Load Raw Data

The pipeline expects the four uncleaned source files.

In [2]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required raw input file: {path}")
    return pd.read_csv(path)

customers_raw = load_csv(RAW_FILES["customers"])
couriers_raw = load_csv(RAW_FILES["couriers"])
deliveries_raw = load_csv(RAW_FILES["deliveries"])
incidents_raw = load_csv(RAW_FILES["incidents"])

raw_summary = pd.DataFrame([
    {"dataset": "customers", "rows": len(customers_raw), "columns": customers_raw.shape[1]},
    {"dataset": "couriers", "rows": len(couriers_raw), "columns": couriers_raw.shape[1]},
    {"dataset": "deliveries", "rows": len(deliveries_raw), "columns": deliveries_raw.shape[1]},
    {"dataset": "incidents", "rows": len(incidents_raw), "columns": incidents_raw.shape[1]},
])
raw_summary

,dataset,rows,columns
0,customers,120,8
1,couriers,65,5
2,deliveries,100110,8
3,incidents,22390,5


## 3. Validate Required Columns

Fail early if a source table is missing fields needed for cleaning or feature engineering.

In [3]:
REQUIRED_COLUMNS = {
    "customers": [
        "customer_id", "customer_name", "customer_size", "city", "signup_date",
        "account_manager", "industry", "payment_terms_days"
    ],
    "couriers": [
        "courier_id", "hire_date", "employment_type", "city", "shift_pattern"
    ],
    "deliveries": [
        "delivery_id", "delivery_date", "customer_id", "courier_id", "city",
        "time_taken_minutes", "delivery_status", "revenue_gbp"
    ],
    "incidents": [
        "incident_id", "delivery_id", "incident_date", "incident_type", "resolution_status"
    ],
}

raw_tables = {
    "customers": customers_raw,
    "couriers": couriers_raw,
    "deliveries": deliveries_raw,
    "incidents": incidents_raw,
}

missing = {
    name: sorted(set(cols) - set(raw_tables[name].columns))
    for name, cols in REQUIRED_COLUMNS.items()
}
missing = {name: cols for name, cols in missing.items() if cols}
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required column validation passed.")

Required column validation passed.


## 4. Clean Source Data

Applies all local cleaning decisions: city standardisation, date parsing, delivery deduplication, missing courier handling, numeric coercion, and unmatched incident flagging.

In [4]:
def parse_mixed_dates(series: pd.Series) -> pd.Series:
    """Parse ISO and UK-style dates, preserving unparseable values as NaT."""
    parsed = pd.to_datetime(series, errors="coerce")
    failed = parsed.isna() & series.notna()
    if failed.any():
        parsed_dayfirst = pd.to_datetime(series[failed], errors="coerce", dayfirst=True)
        parsed.loc[failed] = parsed_dayfirst
    return parsed


def standardise_city(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.title()


customers = customers_raw.copy()
couriers = couriers_raw.copy()
deliveries = deliveries_raw.copy()
incidents = incidents_raw.copy()

# Standardise city values.
for df in [customers, couriers, deliveries]:
    df["city"] = standardise_city(df["city"])

# Parse dates, including mixed incident date formats.
customers["signup_date"] = parse_mixed_dates(customers["signup_date"])
couriers["hire_date"] = parse_mixed_dates(couriers["hire_date"])
deliveries["delivery_date"] = parse_mixed_dates(deliveries["delivery_date"])
incidents["incident_date"] = parse_mixed_dates(incidents["incident_date"])

# Coerce numeric fields needed for modelling.
customers["payment_terms_days"] = pd.to_numeric(customers["payment_terms_days"], errors="coerce")
deliveries["time_taken_minutes"] = pd.to_numeric(deliveries["time_taken_minutes"], errors="coerce")
deliveries["revenue_gbp"] = pd.to_numeric(deliveries["revenue_gbp"], errors="coerce")

# Strip string categories used in joins and conditional aggregations.
for df, cols in [
    (customers, ["customer_id", "customer_name", "customer_size", "account_manager", "industry"]),
    (couriers, ["courier_id", "employment_type", "shift_pattern"]),
    (deliveries, ["delivery_id", "customer_id", "courier_id", "delivery_status"]),
    (incidents, ["incident_id", "delivery_id", "incident_type", "resolution_status"]),
]:
    for col in cols:
        df[col] = df[col].astype("string").str.strip()

# Remove duplicated delivery records, keeping the first canonical row.
duplicate_delivery_rows = int(deliveries["delivery_id"].duplicated().sum())
deliveries = deliveries.drop_duplicates(subset=["delivery_id"], keep="first").copy()

# Preserve deliveries with missing courier assignments as subcontractor/unknown work.
missing_courier_rows = int(deliveries["courier_id"].isna().sum())
deliveries["courier_id"] = deliveries["courier_id"].fillna("SUBCONTRACTOR_UNKNOWN")

# Flag unmatched incident references after delivery deduplication.
incidents["is_unmatched_delivery_id"] = ~incidents["delivery_id"].isin(deliveries["delivery_id"])
unmatched_incident_rows = int(incidents["is_unmatched_delivery_id"].sum())

cleaning_summary = pd.DataFrame([
    {"cleaning_step": "duplicate delivery_id rows removed", "rows_affected": duplicate_delivery_rows},
    {"cleaning_step": "missing courier_id values labelled SUBCONTRACTOR_UNKNOWN", "rows_affected": missing_courier_rows},
    {"cleaning_step": "incident delivery_id values flagged as unmatched", "rows_affected": unmatched_incident_rows},
])

print("Cleaned shapes:")
print("customers", customers.shape)
print("couriers", couriers.shape)
print("deliveries", deliveries.shape)
print("incidents", incidents.shape)
cleaning_summary

Cleaned shapes:
customers (120, 8)
couriers (65, 5)
deliveries (98148, 8)
incidents (22390, 6)


,cleaning_step,rows_affected
0,duplicate delivery_id rows removed,1962
1,missing courier_id values labelled SUBCONTRACT...,3913
2,incident delivery_id values flagged as unmatched,5


## 5. Snapshot and Aggregation Helpers

Each model row is one customer at one month-end snapshot. Features use the previous 60 days; the target uses the following 60 days.

In [5]:
def safe_divide(numerator, denominator):
    """Vectorised divide that returns 0 where denominator is 0 or missing."""
    numerator = pd.Series(numerator).astype(float)
    denominator = pd.Series(denominator).astype(float)
    result = np.where((denominator == 0) | denominator.isna(), 0, numerator / denominator)
    return result


def generate_snapshot_dates(deliveries_df: pd.DataFrame, config: dict) -> pd.DatetimeIndex:
    min_date = deliveries_df["delivery_date"].min()
    max_date = deliveries_df["delivery_date"].max()
    lookback = pd.Timedelta(days=config["lookback_days"])
    horizon = pd.Timedelta(days=config["prediction_horizon_days"])

    earliest_snapshot = min_date + lookback
    latest_snapshot = max_date - horizon

    if pd.isna(earliest_snapshot) or pd.isna(latest_snapshot) or earliest_snapshot > latest_snapshot:
        raise ValueError("Not enough valid delivery dates to create leakage-safe snapshot windows.")

    if config["snapshot_frequency"] != "month_end":
        raise ValueError("Only snapshot_frequency='month_end' is implemented.")

    return pd.date_range(
        earliest_snapshot.normalize(),
        latest_snapshot.normalize(),
        freq=pd.offsets.MonthEnd()
    )


def aggregate_deliveries_window(deliveries_df: pd.DataFrame, start_date, end_date, suffix: str) -> pd.DataFrame:
    mask = (deliveries_df["delivery_date"] >= start_date) & (deliveries_df["delivery_date"] < end_date)
    d = deliveries_df.loc[mask].copy()

    columns = [
        "customer_id", f"deliveries_{suffix}", f"revenue_{suffix}",
        f"failed_deliveries_{suffix}", f"returned_deliveries_{suffix}",
        f"delivered_deliveries_{suffix}", f"avg_delivery_time_{suffix}",
    ]
    if d.empty:
        return pd.DataFrame(columns=columns)

    status = d["delivery_status"].astype("string").str.strip().str.casefold()
    d["is_failed"] = status.eq("failed").astype(int)
    d["is_returned"] = status.eq("returned").astype(int)
    d["is_delivered"] = status.eq("delivered").astype(int)

    return d.groupby("customer_id", as_index=False).agg(
        **{
            f"deliveries_{suffix}": ("delivery_id", "count"),
            f"revenue_{suffix}": ("revenue_gbp", "sum"),
            f"failed_deliveries_{suffix}": ("is_failed", "sum"),
            f"returned_deliveries_{suffix}": ("is_returned", "sum"),
            f"delivered_deliveries_{suffix}": ("is_delivered", "sum"),
            f"avg_delivery_time_{suffix}": ("time_taken_minutes", "mean"),
        }
    )


def aggregate_incidents_window(deliveries_df: pd.DataFrame, incidents_df: pd.DataFrame, start_date, end_date, suffix: str) -> pd.DataFrame:
    valid_incidents = incidents_df.loc[~incidents_df["is_unmatched_delivery_id"]].copy()
    incident_base = valid_incidents.merge(
        deliveries_df[["delivery_id", "customer_id"]],
        on="delivery_id",
        how="left",
        validate="many_to_one"
    )

    mask = (incident_base["incident_date"] >= start_date) & (incident_base["incident_date"] < end_date)
    x = incident_base.loc[mask].copy()

    columns = [
        "customer_id", f"incident_count_{suffix}", f"complaint_count_{suffix}",
        f"late_delivery_count_{suffix}", f"damaged_parcel_count_{suffix}", f"lost_parcel_count_{suffix}",
    ]
    if x.empty:
        return pd.DataFrame(columns=columns)

    incident_type = x["incident_type"].astype("string").str.strip().str.casefold()
    x["is_complaint"] = incident_type.eq("customer complaint").astype(int)
    x["is_late_delivery"] = incident_type.eq("late delivery").astype(int)
    x["is_damaged_parcel"] = incident_type.eq("damaged parcel").astype(int)
    x["is_lost_parcel"] = incident_type.eq("lost parcel").astype(int)

    return x.groupby("customer_id", as_index=False).agg(
        **{
            f"incident_count_{suffix}": ("incident_id", "count"),
            f"complaint_count_{suffix}": ("is_complaint", "sum"),
            f"late_delivery_count_{suffix}": ("is_late_delivery", "sum"),
            f"damaged_parcel_count_{suffix}": ("is_damaged_parcel", "sum"),
            f"lost_parcel_count_{suffix}": ("is_lost_parcel", "sum"),
        }
    )

## 6. Build Customer Snapshots

Historical features are calculated from data available before the snapshot. Future values are created only to generate the target and are dropped before export.

In [6]:
def build_customer_snapshot(snapshot_date, customers_df, deliveries_df, incidents_df, config):
    lookback = pd.Timedelta(days=config["lookback_days"])
    horizon = pd.Timedelta(days=config["prediction_horizon_days"])
    half_lookback = pd.Timedelta(days=config["lookback_days"] // 2)

    feature_start = snapshot_date - lookback
    feature_midpoint = snapshot_date - half_lookback
    feature_end = snapshot_date
    future_start = snapshot_date
    future_end = snapshot_date + horizon

    base = customers_df[[
        "customer_id", "customer_name", "customer_size", "city", "signup_date",
        "account_manager", "industry", "payment_terms_days"
    ]].copy()
    base["snapshot_date"] = snapshot_date

    hist_deliv = aggregate_deliveries_window(deliveries_df, feature_start, feature_end, "lookback_60d")
    hist_inc = aggregate_incidents_window(deliveries_df, incidents_df, feature_start, feature_end, "lookback_60d")
    prev_30d = aggregate_deliveries_window(deliveries_df, feature_start, feature_midpoint, "prev_30d")
    latest_30d = aggregate_deliveries_window(deliveries_df, feature_midpoint, feature_end, "latest_30d")
    future_deliv = aggregate_deliveries_window(deliveries_df, future_start, future_end, "future_60d")

    out = base.merge(hist_deliv, on="customer_id", how="left")
    out = out.merge(hist_inc, on="customer_id", how="left")
    out = out.merge(prev_30d[["customer_id", "deliveries_prev_30d", "revenue_prev_30d"]], on="customer_id", how="left")
    out = out.merge(latest_30d[["customer_id", "deliveries_latest_30d", "revenue_latest_30d"]], on="customer_id", how="left")
    out = out.merge(future_deliv[["customer_id", "deliveries_future_60d", "revenue_future_60d"]], on="customer_id", how="left")

    numeric_cols = out.select_dtypes(include=["number"]).columns
    out[numeric_cols] = out[numeric_cols].fillna(0)
    out["customer_tenure_days"] = (out["snapshot_date"] - out["signup_date"]).dt.days
    return out

snapshot_dates = generate_snapshot_dates(deliveries, CHURN_CONFIG)
customer_snapshots = pd.concat(
    [build_customer_snapshot(dt, customers, deliveries, incidents, CHURN_CONFIG) for dt in snapshot_dates],
    ignore_index=True
)

print(f"Snapshot dates: {list(snapshot_dates.strftime('%Y-%m-%d'))}")
print("Customer snapshots:", customer_snapshots.shape)

Snapshot dates: ['2024-11-30', '2024-12-31', '2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30']
Customer snapshots: (720, 27)


## 7. Engineer Target and Core Features

The target is based on future relative decline. All future-derived fields are kept only temporarily and excluded from the exported model table.

In [7]:
# Future decline metrics for label generation only.
customer_snapshots["delivery_volume_decline_pct"] = safe_divide(
    customer_snapshots["deliveries_lookback_60d"] - customer_snapshots["deliveries_future_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["revenue_decline_pct"] = safe_divide(
    customer_snapshots["revenue_lookback_60d"] - customer_snapshots["revenue_future_60d"],
    customer_snapshots["revenue_lookback_60d"]
)

market_decline = customer_snapshots.groupby("snapshot_date", as_index=False).agg(
    market_deliveries_lookback_60d=("deliveries_lookback_60d", "sum"),
    market_deliveries_future_60d=("deliveries_future_60d", "sum"),
    market_revenue_lookback_60d=("revenue_lookback_60d", "sum"),
    market_revenue_future_60d=("revenue_future_60d", "sum"),
)
market_decline["market_volume_decline_pct"] = safe_divide(
    market_decline["market_deliveries_lookback_60d"] - market_decline["market_deliveries_future_60d"],
    market_decline["market_deliveries_lookback_60d"]
)
market_decline["market_revenue_decline_pct"] = safe_divide(
    market_decline["market_revenue_lookback_60d"] - market_decline["market_revenue_future_60d"],
    market_decline["market_revenue_lookback_60d"]
)
customer_snapshots = customer_snapshots.merge(
    market_decline[["snapshot_date", "market_volume_decline_pct", "market_revenue_decline_pct"]],
    on="snapshot_date",
    how="left"
)
customer_snapshots["relative_volume_decline_pct"] = (
    customer_snapshots["delivery_volume_decline_pct"] - customer_snapshots["market_volume_decline_pct"]
)
customer_snapshots["relative_revenue_decline_pct"] = (
    customer_snapshots["revenue_decline_pct"] - customer_snapshots["market_revenue_decline_pct"]
)

# Leakage-safe historical features.
customer_snapshots["avg_revenue_per_delivery_lookback_60d"] = safe_divide(
    customer_snapshots["revenue_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["failed_delivery_rate_lookback_60d"] = safe_divide(
    customer_snapshots["failed_deliveries_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["return_rate_lookback_60d"] = safe_divide(
    customer_snapshots["returned_deliveries_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["success_rate_lookback_60d"] = safe_divide(
    customer_snapshots["delivered_deliveries_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["incident_rate_lookback_60d"] = safe_divide(
    customer_snapshots["incident_count_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["complaint_rate_lookback_60d"] = safe_divide(
    customer_snapshots["complaint_count_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["late_delivery_rate_lookback_60d"] = safe_divide(
    customer_snapshots["late_delivery_count_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["damaged_parcel_rate_lookback_60d"] = safe_divide(
    customer_snapshots["damaged_parcel_count_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)
customer_snapshots["lost_parcel_rate_lookback_60d"] = safe_divide(
    customer_snapshots["lost_parcel_count_lookback_60d"],
    customer_snapshots["deliveries_lookback_60d"]
)

print("Core feature engineering complete.")

Core feature engineering complete.


## 8. Engineer Relative and Industry Features

These features compare a customer or industry to the wider UrbanShift customer base at the same snapshot.

In [8]:
snapshot_medians = customer_snapshots.groupby("snapshot_date", as_index=False).agg(
    median_deliveries_60d=("deliveries_lookback_60d", "median"),
    median_revenue_60d=("revenue_lookback_60d", "median"),
)
customer_snapshots = customer_snapshots.merge(snapshot_medians, on="snapshot_date", how="left")
customer_snapshots["relative_volume_strength"] = safe_divide(
    customer_snapshots["deliveries_lookback_60d"],
    customer_snapshots["median_deliveries_60d"]
)
customer_snapshots["volume_growth_lookback"] = safe_divide(
    customer_snapshots["deliveries_latest_30d"] - customer_snapshots["deliveries_prev_30d"],
    customer_snapshots["deliveries_prev_30d"]
)
market_growth = customer_snapshots.groupby("snapshot_date", as_index=False).agg(
    market_deliveries_prev_30d=("deliveries_prev_30d", "sum"),
    market_deliveries_latest_30d=("deliveries_latest_30d", "sum"),
    market_revenue_prev_30d=("revenue_prev_30d", "sum"),
    market_revenue_latest_30d=("revenue_latest_30d", "sum"),
)
market_growth["market_volume_growth_lookback"] = safe_divide(
    market_growth["market_deliveries_latest_30d"] - market_growth["market_deliveries_prev_30d"],
    market_growth["market_deliveries_prev_30d"]
)
market_growth["market_revenue_growth_lookback"] = safe_divide(
    market_growth["market_revenue_latest_30d"] - market_growth["market_revenue_prev_30d"],
    market_growth["market_revenue_prev_30d"]
)
customer_snapshots = customer_snapshots.merge(
    market_growth[["snapshot_date", "market_volume_growth_lookback", "market_revenue_growth_lookback"]],
    on="snapshot_date",
    how="left"
)
customer_snapshots["relative_volume_growth_lookback"] = (
    customer_snapshots["volume_growth_lookback"] - customer_snapshots["market_volume_growth_lookback"]
)
industry_snapshot = customer_snapshots.groupby(["snapshot_date", "industry"], as_index=False).agg(
    industry_revenue=("revenue_lookback_60d", "sum")
)
market_snapshot = customer_snapshots.groupby("snapshot_date", as_index=False).agg(
    total_revenue=("revenue_lookback_60d", "sum")
)
industry_snapshot = industry_snapshot.merge(market_snapshot, on="snapshot_date", how="left")
industry_snapshot["industry_revenue_share"] = safe_divide(
    industry_snapshot["industry_revenue"],
    industry_snapshot["total_revenue"]
)
industry_snapshot = industry_snapshot.sort_values(["industry", "snapshot_date"])
industry_snapshot["industry_revenue_growth_pct"] = industry_snapshot.groupby("industry")["industry_revenue"].pct_change()
market_snapshot = market_snapshot.sort_values("snapshot_date")
market_snapshot["market_revenue_growth_pct"] = market_snapshot["total_revenue"].pct_change()
industry_snapshot = industry_snapshot.merge(
    market_snapshot[["snapshot_date", "market_revenue_growth_pct"]],
    on="snapshot_date",
    how="left"
)
industry_snapshot["industry_relative_revenue_strength"] = (
    industry_snapshot["industry_revenue_growth_pct"] - industry_snapshot["market_revenue_growth_pct"]
)
customer_snapshots = customer_snapshots.merge(
    industry_snapshot[["snapshot_date", "industry", "industry_revenue_share", "industry_relative_revenue_strength"]],
    on=["snapshot_date", "industry"],
    how="left"
)

print("Relative and industry feature engineering complete.")

Relative and industry feature engineering complete.


## 9. Create Target and Final Model Table

The exported table keeps only approved model columns plus `customer_id` for traceability and `at_risk` as the target.

In [9]:
customer_snapshots["eligible_for_training"] = (
    customer_snapshots["deliveries_lookback_60d"] >= CHURN_CONFIG["min_baseline_deliveries"]
)

customer_snapshots["at_risk"] = (
    customer_snapshots["eligible_for_training"]
    & (customer_snapshots["relative_volume_decline_pct"] >= CHURN_CONFIG["relative_volume_drop"])
    & (customer_snapshots["relative_revenue_decline_pct"] >= CHURN_CONFIG["relative_revenue_drop"])
).astype(int)

feature_cols = [
    "snapshot_date",
    "customer_size",
    "city",
    "industry",
    "industry_revenue_share",
    "industry_relative_revenue_strength",
    "payment_terms_days",
    "customer_tenure_days",
    "avg_revenue_per_delivery_lookback_60d",
    "relative_volume_strength",
    "volume_growth_lookback",
    "relative_volume_growth_lookback",
    "failed_delivery_rate_lookback_60d",
    "return_rate_lookback_60d",
    "avg_delivery_time_lookback_60d",
    "incident_rate_lookback_60d",
    "complaint_rate_lookback_60d",
    "late_delivery_rate_lookback_60d",
    "damaged_parcel_rate_lookback_60d",
    "lost_parcel_rate_lookback_60d",
]
target_col = "at_risk"

leakage_or_audit_cols = [
    "deliveries_future_60d", "revenue_future_60d",
    "delivery_volume_decline_pct", "revenue_decline_pct",
    "market_volume_decline_pct", "market_revenue_decline_pct",
    "relative_volume_decline_pct", "relative_revenue_decline_pct",
        "eligible_for_training",
]
forbidden_model_inputs = set(leakage_or_audit_cols + [
    "customer_name", "account_manager",
    "volume_decline_pct", "relative_volume_growth", "relative_revenue_growth",
    "deliveries_lookback_60d", "revenue_lookback_60d",
    "relative_revenue_strength", "revenue_growth_lookback", "relative_revenue_growth_lookback",
    "success_rate_lookback_60d", "incident_count_lookback_60d",
])
accidental_leakage_cols = sorted(forbidden_model_inputs.intersection(feature_cols))
if accidental_leakage_cols:
    raise ValueError(f"Potential leakage/redundant columns included in feature_cols: {accidental_leakage_cols}")

missing_features = [col for col in feature_cols if col not in customer_snapshots.columns]
if missing_features:
    raise ValueError(f"Expected feature columns are missing: {missing_features}")

model_table = customer_snapshots.loc[
    customer_snapshots["eligible_for_training"],
    ["customer_id"] + feature_cols + [target_col]
].copy()

# First industry growth point is naturally null; use neutral value rather than dropping rows.
model_table["industry_relative_revenue_strength"] = model_table["industry_relative_revenue_strength"].fillna(0)

print("Model table shape:", model_table.shape)
print(f"Positive class rate: {model_table[target_col].mean():.2%}")
model_table.head()

Model table shape: (714, 22)
Positive class rate: 8.68%


,customer_id,snapshot_date,customer_size,city,industry,industry_revenue_share,industry_relative_revenue_strength,payment_terms_days,customer_tenure_days,avg_revenue_per_delivery_lookback_60d,relative_volume_strength,volume_growth_lookback,relative_volume_growth_lookback,failed_delivery_rate_lookback_60d,return_rate_lookback_60d,avg_delivery_time_lookback_60d,incident_rate_lookback_60d,complaint_rate_lookback_60d,late_delivery_rate_lookback_60d,damaged_parcel_rate_lookback_60d,lost_parcel_rate_lookback_60d,at_risk
0,CUST1000,2024-11-30,Mid-size Retailer,London,Beauty,0.123535,0.0,60,901,5.948517,2.484211,-0.081301,-0.146712,0.042373,0.016949,72.724576,0.169492,0.038136,0.016949,0.046610,0.012712,0
1,CUST1001,2024-11-30,Mid-size Retailer,London,Books,0.088978,0.0,30,199,6.066206,3.884211,-0.016129,-0.081540,0.037940,0.027100,70.051491,0.176152,0.029810,0.032520,0.040650,0.016260,0
2,CUST1002,2024-11-30,Small Retailer,Bristol,Fashion,0.363793,0.0,30,759,4.657647,0.536842,-0.038462,-0.103873,0.078431,0.058824,65.039216,0.294118,0.039216,0.098039,0.058824,0.058824,0
3,CUST1003,2024-11-30,Small Retailer,Leeds,Beauty,0.123535,0.0,60,155,4.654000,0.368421,-0.157895,-0.223306,0.028571,0.057143,66.514286,0.200000,0.057143,0.057143,0.028571,0.000000,0
4,CUST1004,2024-11-30,Small Retailer,Manchester,Electronics,0.128233,0.0,30,886,4.626983,1.221053,-0.066667,-0.132078,0.051724,0.034483,66.025862,0.284483,0.034483,0.068966,0.068966,0.025862,0


## 10. Save Final Output

Only the final churn model table is written to disk.

In [10]:
model_output_path = OUTPUT_DIR / "customer_churn_model_table.csv"
model_table.to_csv(model_output_path, index=False)
print("Saved:", model_output_path.resolve())

Saved: C:\Users\Ian_C\OneDrive\Desktop\Work\Project 4\urbanshift-bristol\code\outputs\customer_churn_model_table.csv
